In [ ]:
import os
from transformers import VisionEncoderDecoderModel
from transformers import DonutProcessor # 혹은 직접 임포트한 커스텀 processor
import torch
import json
from PIL import Image
from pdf2image import convert_from_path

In [ ]:
#Fine-tuning 모델 Load

model_path = "outputs/donut_finetuned"

processor = DonutProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)

#❗ pad_token 설정 (반드시!)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    
model.eval()  # 추론 모드로 변경
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
print("pad_token:", processor.tokenizer.pad_token)
print("pad_token_id  :", processor.tokenizer.pad_token_id)   # 정수


In [ ]:
# PDF가 들어있는 폴더
pdf_folder = r"data/raw_pdf"
# 이미지 저장 폴더 (같은 경로에 "converted_images" 폴더 생성)
image_output_folder = os.path.join(pdf_folder, "converted_images")
os.makedirs(image_output_folder, exist_ok=True)

for file_name in os.listdir(pdf_folder):
    if file_name.lower().endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, file_name)
        try:
            pages = convert_from_path(pdf_path, dpi=300, first_page=1, last_page=1)
            image = pages[0].convert("RGB")

            # 저장 경로 설정 (.pdf → .png 이름 변환)
            base_name = os.path.splitext(file_name)[0]
            image_path = os.path.join(image_output_folder, f"{base_name}.png")
            image.save(image_path)

            print(f"✅ 이미지 저장 완료: {image_path}")
        except Exception as e:
            print(f"❌ 변환 실패: {file_name} | 에러: {e}")

In [ ]:
import os
import json
import torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

# 1. 설정
image_folder = r"data/images"
model_path = "outputs/donut_finetuned"
device = "cuda" if torch.cuda.is_available() else "cpu"
task_prompt = "<s_gt_parse>"

# 2. 모델과 processor 불러오기
processor = DonutProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)
model.eval()

# 3. 이미지 추론 루프
for file_name in os.listdir(image_folder):
    if file_name.lower().endswith(".png"):
        image_path = os.path.join(image_folder, file_name)
        try:
            # (1) 이미지 로드 및 전처리
            image = Image.open(image_path).convert("RGB")
            pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

            # (2) 디코더 입력 준비
            decoder_input_ids = processor.tokenizer(
                task_prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).input_ids.to(device)

            # (3) 추론
            outputs = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=512,
                early_stopping=True,
                num_beams=5,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id
            )

            # (4) 디코딩
            prediction = processor.batch_decode(outputs, skip_special_tokens=True)[0]
            print(f"📄 {file_name} → 추론 결과:", prediction)

            # (5) JSON 파싱 시도
            try:
                json_text = prediction.replace("<s_gt_parse>", "").replace("</s>", "").strip()
                parsed = json.loads(json_text)
                print("🧾 파싱된 JSON 결과:", parsed)
            except Exception as je:
                print("⚠️ JSON 파싱 실패:", je)

        except Exception as e:
            print(f"❌ {file_name} 처리 중 에러: {e}")


In [ ]:
 # ✅ 2. 프롬프트 텍스트 토크나이징
decoder_input_ids = processor.tokenizer(
task_prompt,
return_tensors="pt",
padding="max_length",  # ← 여기 padding 사용
truncation=True,
max_length=512
).input_ids.to(device)

In [ ]:
# GPU 사용 가능시 GPU로 올리기
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
pixel_values = pixel_values.to(device)

# 추론
task_prompt = "<s_gt_parse>"  # 학습 시 사용했던 동일한 프롬프트
decoder_input_ids = processor.tokenizer(task_prompt, return_tensors="pt").input_ids.to(device)

# generate() 호출
outputs = model.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=512,
    early_stopping=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id
)

In [ ]:
# 디코딩
prediction = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print("추론 결과 (JSON 문자열):", prediction)

# JSON 파싱
parsed_result = json.loads(prediction.replace("<s_gt_parse>", "").replace("</s>", ""))
print("파싱된 JSON 결과:", parsed_result)
